In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score)

In [3]:
df = pd.read_parquet("../data/processed/df_combined_eda_sample.parquet")

In [4]:
df["merchant_state"] = df["merchant_state"].fillna("Unknown")
df["errors"] = df["errors"].fillna("no_error")

In [10]:
drop_cols = ['id', 'client_id', 'merchant_id']
df = df.drop(columns=drop_cols, errors='ignore')

In [15]:
df['weekday'] = df['date'].dt.weekday

In [20]:
np.isinf(df.select_dtypes(include='number')).sum()

amount                        0
current_age                   0
retirement_age                0
birth_year                    0
birth_month                   0
latitude                      0
longitude                     0
credit_score                  0
num_credit_cards              0
card_number                   0
cvv                           0
num_cards_issued              0
credit_limit                  0
year_pin_last_changed         0
is_fraud                      0
year                          0
month                         0
day                           0
hour                          0
time_diff                     0
txn_per_day                   0
avg_amount_user               0
log_amount                    0
amount_deviation              0
amount_zscore                 0
high_amount                   0
is_foreign_city               0
txn_last_1h                   0
fast_tx                       0
very_fast_tx                  0
unique_merchants_last10       0
credit_u

In [21]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [22]:
#Train / Test split
X = df.drop(columns=["is_fraud"])
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train fraud oranı:", y_train.mean())
print("Test fraud oranı:", y_test.mean())

Train shape: (268000, 63)
Test shape: (67000, 63)
Train fraud oranı: 0.0009962686567164178
Test fraud oranı: 0.001


In [17]:

num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
#cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
cat_cols = [
    col for col in X_train.select_dtypes(include='object').columns
    if X_train[col].nunique() < 50
]
cat_cols += ['hour', 'weekday', 'is_night']
exclude_cols = [
    'client_id',
    'merchant_id',
    'card_number',
    'id'
]

exclude_cols_num = [
    'client_id',
    'merchant_id',
    'card_number',
    'id',
    'weekday'
    'is_fraud'
]
cat_cols = [col for col in cat_cols if col not in exclude_cols]
num_cols = [col for col in num_cols if col not in exclude_cols_num]

#num_cols_all = df.select_dtypes(include=[np.number]).columns
X_train[num_cols] = X_train[num_cols].fillna(0)

print(X_train.isna().sum().sort_values(ascending=False).head(20))
print("Numerik kolon sayısı:", len(num_cols))
print("Kategorik kolon sayısı:", len(cat_cols))
print("Kategorik kolonlar:", cat_cols)
print("Nümerik kolonlar:", num_cols)

date                0
amount_zscore       0
card_on_dark_web    0
mcc_description     0
year                0
month               0
day                 0
hour                0
day_of_week         0
is_weekend          0
time_diff           0
txn_per_day         0
avg_amount_user     0
log_amount          0
amount_deviation    0
high_amount         0
acct_open_date      0
home_city           0
is_foreign_city     0
txn_last_1h         0
dtype: int64
Numerik kolon sayısı: 36
Kategorik kolon sayısı: 12
Kategorik kolonlar: ['use_chip', 'errors', 'gender', 'card_brand', 'card_type', 'has_chip', 'card_on_dark_web', 'day_of_week', 'card_darkweb_flag', 'hour', 'weekday', 'is_night']
Nümerik kolonlar: ['amount', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'latitude', 'longitude', 'credit_score', 'num_credit_cards', 'cvv', 'num_cards_issued', 'credit_limit', 'year_pin_last_changed', 'year', 'month', 'day', 'hour', 'time_diff', 'txn_per_day', 'avg_amount_user', 'log_amount', 'am

In [18]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

In [23]:
#MVP modelini eğitelim
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)
print("Model eğitildi.")

Model eğitildi.


In [24]:
#Modeli Değerlendirme
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nPrecision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Average Precision:", average_precision_score(y_test, y_proba))

#buradaki amaç modelimiz gerçekten fraud yakalıyor mu ? 


Confusion Matrix:
[[58288  8645]
 [    8    59]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.87      0.93     66933
           1       0.01      0.88      0.01        67

    accuracy                           0.87     67000
   macro avg       0.50      0.88      0.47     67000
weighted avg       1.00      0.87      0.93     67000


Precision: 0.006778492647058824
Recall: 0.8805970149253731
F1: 0.013453426063162695
ROC AUC: 0.9443135494594616
Average Precision: 0.038577251576929004


In [26]:
joblib.dump(model, "../models/behaviour_mvp_model.joblib")


['../models/behaviour_mvp_model.joblib']